# AdaBoost von Grund auf

Dies ist ein **optionales** Notebook. Bisher haben wir Modelle kombiniert, die unabhängig voneinander trainiert und dann zusammengelegt wurden (Voting, Mittelung, Stacking). *Boosting* geht einen anderen Weg: Es trainiert Weak Learner nacheinander, und jeder neue Learner konzentriert sich auf die Punkte, die die vorherigen falsch klassifiziert haben. AdaBoost („adaptive boosting“) ist das klassische Beispiel. Hier liest du eine kompakte Implementierung von Grund auf durch und beobachtest, wie die Fehlerrate sinkt, je mehr Weak Learner hinzukommen.

## Lernziele

Am Ende dieses Notebooks kannst du:

- nachvollziehen, wie AdaBoost Weak Learner nacheinander trainiert und sich jeder dabei auf die Fehler der vorherigen Runde konzentriert.
- die Aktualisierung in AdaBoost berechnen: jedem Learner einen Score zuweisen und die falsch klassifizierten Punkte neu gewichten.
- eine von Grund auf geschriebene AdaBoost-Implementierung auf einen Benchmark-Datensatz anwenden und den Trainings- und Testfehler messen.
- visualisieren, wie die Fehlerrate sinkt, wenn die Anzahl der Boosting-Runden wächst.

Den folgenden Code haben wir uns freundlich bei [Jaime Pastor](https://github.com/jaimeps/adaboost-implementation/blob/master/adaboost.py) ausgeliehen. 

Geh den Code durch und versuch zu verstehen, was jede Funktion macht. Vergiss nicht, ein paar Kommentare zu schreiben oder den Funktionen einen Docstring hinzuzufügen. 

## Wie AdaBoost funktioniert

Ein einzelner Entscheidungsstumpf (ein Baum der Tiefe 1) ist ein *Weak Learner*: Bei einem schwierigen Problem ist er nur ein wenig besser als Raten. AdaBoost macht aus einer Folge solcher Weak Learner einen Strong Learner. Es hält für jeden Trainingspunkt ein Gewicht fest. In jeder Runde trainiert es einen Stumpf auf den gewichteten Daten, gibt diesem Stumpf ein Stimmgewicht (`alpha`), das davon abhängt, wie treffsicher er war, und erhöht dann die Gewichte der Punkte, die der Stumpf falsch klassifiziert hat, damit der nächste Stumpf ihnen mehr Aufmerksamkeit schenkt. Die finale Vorhersage ist eine gewichtete Abstimmung aller Stümpfe.

Das ist der umgekehrte Trade-off zum Bagging (das ein Random Forest verwendet), bei dem viele Bäume unabhängig und parallel trainiert werden, um die Varianz zu verringern. Boosting trainiert seine Learner nacheinander, um den Bias zu verringern, und kann so flache Stümpfe mit hohem Bias in ein treffsicheres Modell verwandeln.




```mermaid
flowchart TD
    A["Start: gleiche Gewichte für jeden Trainingspunkt"] --> B["Weak Learner (einen Entscheidungsstumpf) auf den gewichteten Daten trainieren"]
    B --> C["Gewichteten Fehler messen und Stimmgewicht (alpha) berechnen"]
    C --> D["Gewichte der falsch klassifizierten Punkte erhöhen"]
    D --> E{"M Runden erledigt?"}
    E -->|"Nein"| B
    E -->|"Ja"| F["Finales Modell: gewichtete Abstimmung aller Weak Learner"]
```


## Setup

Importiere die Bibliotheken. `make_hastie_10_2` erzeugt ein binäres Klassifikationsproblem, das für einen einzelnen Stumpf absichtlich schwer ist. So lässt sich der Effekt von Boosting leicht erkennen.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    make_hastie_10_2,
)  # make_hastie_10_2 erzeugt einen Datensatz für die binäre Klassifikation
import matplotlib.pyplot as plt

## Hilfsfunktionen

Zwei kleine Hilfsfunktionen messen die Fehlerrate (den Anteil falsch klassifizierter Punkte) und geben sie aus, und eine dritte trainiert einen beliebigen einzelnen Klassifikator und berechnet seinen Score, als Baseline zum Vergleich.

In [ ]:
""" HILFSFUNKTION: FEHLERRATE BERECHNEN ====================================="""


def get_error_rate(pred, Y):
    return sum(pred != Y) / float(len(Y))


""" HILFSFUNKTION: FEHLERRATE AUSGEBEN ======================================"""


def print_error_rate(err):
    print("Error rate: Training: %.4f - Test: %.4f" % err)


""" HILFSFUNKTION: GENERISCHER KLASSIFIKATOR ================================"""


def generic_clf(Y_train, X_train, Y_test, X_test, clf):
    clf.fit(X_train, Y_train)
    pred_train = clf.predict(X_train)
    pred_test = clf.predict(X_test)
    return get_error_rate(pred_train, Y_train), get_error_rate(pred_test, Y_test)

## Der AdaBoost-Algorithmus

Das ist das Herzstück des Notebooks. Lies den Code zusammen mit dem Schleifendiagramm oben: `w` enthält die Gewichte pro Punkt, `err_m` ist der gewichtete Fehler des aktuellen Stumpfs, `alpha_m` ist sein Stimmgewicht in der finalen Abstimmung, und in der Zeile `w = w * exp(...)` werden die Gewichte der falsch klassifizierten Punkte erhöht. Die laufenden `pred_train` und `pred_test` summieren die gewichteten Stimmen auf, und `np.sign` macht aus dem aufsummierten Score ein Klassenlabel.

Die leeren `#`-Zeilen sind Platzhalter für deine eigenen Kommentare, wie es die Aufgabe oben vorschlägt.

In [ ]:
""" ADABOOST-IMPLEMENTIERUNG ================================================"""


def adaboost_clf(Y_train, X_train, Y_test, X_test, M, clf):
    n_train, n_test = len(X_train), len(X_test)
    #
    w = np.ones(n_train) / n_train
    pred_train, pred_test = [np.zeros(n_train), np.zeros(n_test)]

    for i in range(M):
        #
        clf.fit(X_train, Y_train, sample_weight=w)
        pred_train_i = clf.predict(X_train)
        pred_test_i = clf.predict(X_test)
        #
        miss = [int(x) for x in (pred_train_i != Y_train)]
        #
        miss2 = [x if x == 1 else -1 for x in miss]
        #
        err_m = np.dot(w, miss) / sum(w)
        #
        alpha_m = 0.5 * np.log((1 - err_m) / float(err_m))
        #
        w = np.multiply(w, np.exp([float(x) * alpha_m for x in miss2]))
        #
        pred_train = [
            sum(x) for x in zip(pred_train, [x * alpha_m for x in pred_train_i])
        ]
        pred_test = [sum(x) for x in zip(pred_test, [x * alpha_m for x in pred_test_i])]

    pred_train, pred_test = np.sign(pred_train), np.sign(pred_test)
    #
    return get_error_rate(pred_train, Y_train), get_error_rate(pred_test, Y_test)

## Fehlerrate darstellen

Eine kleine Hilfsfunktion, die den Trainings- und Testfehler gegen die Anzahl der Boosting-Runden darstellt, mit einer gestrichelten Linie für die Baseline des einzelnen Stumpfs.

In [ ]:
""" PLOT-FUNKTION ==========================================================="""


def plot_error_rate(er_train, er_test):
    df_error = pd.DataFrame([er_train, er_test]).T
    df_error.columns = ["Training", "Test"]
    plot1 = df_error.plot(
        linewidth=3, figsize=(8, 6), color=["lightblue", "darkblue"], grid=True
    )
    plot1.set_xlabel("Number of iterations", fontsize=12)
    plot1.set_xticklabels(range(0, 450, 50))
    plot1.set_ylabel("Error rate", fontsize=12)
    plot1.set_title("Error rate vs number of iterations", fontsize=16)
    plt.axhline(y=er_test[0], linewidth=1, color="red", ls="dashed")

## Experiment durchführen

Wir erzeugen den Hastie-Datensatz, trainieren als Baseline einen einzelnen Entscheidungsstumpf und lassen dann AdaBoost mit einer wachsenden Anzahl an Runden laufen (10, 20, ..., 400), wobei wir jedes Mal den Fehler festhalten. Die Darstellung dieser Werte zeigt, wie sich das Ensemble verbessert, je mehr Weak Learner hinzukommen.

In [ ]:
""" HAUPTSKRIPT ============================================================="""
if __name__ == "__main__":
    # Daten einlesen
    x, y = make_hastie_10_2()
    df = pd.DataFrame(x)
    df["Y"] = y

    # In Trainings- und Testdatensatz aufteilen
    train, test = train_test_split(df, test_size=0.2)
    X_train, Y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, Y_test = test.iloc[:, :-1], test.iloc[:, -1]

    # Zuerst einen einfachen Entscheidungsbaum trainieren
    clf_tree = DecisionTreeClassifier(max_depth=1, random_state=1)
    er_tree = generic_clf(Y_train, X_train, Y_test, X_test, clf_tree)

    # Adaboost-Klassifikator mit einem Entscheidungsbaum als Basis-Estimator trainieren
    # Mit unterschiedlicher Anzahl von Iterationen testen
    er_train, er_test = [er_tree[0]], [er_tree[1]]
    x_range = range(10, 410, 10)
    for i in x_range:
        er_i = adaboost_clf(Y_train, X_train, Y_test, X_test, i, clf_tree)
        er_train.append(er_i[0])
        er_test.append(er_i[1])

    # Fehlerrate vs Anzahl der Iterationen vergleichen
    plot_error_rate(er_train, er_test)

### Ergebnis lesen

Der einzelne Stumpf liegt bei einer Fehlerrate von etwa **0,44**, bei diesem schwierigen Problem also nur wenig besser als ein Münzwurf. Während AdaBoost Runden hinzufügt, sinkt der Fehler stetig: etwa **0,35** nach 10 Runden, **0,20** nach 50 und bis auf etwa **0,10** nach 400 Runden. Trainings- und Testkurve fallen gemeinsam, und die Testkurve bleibt nah an der Trainingskurve. Das zeigt, dass sich AdaBoost wirklich verbessert, statt nur den Trainingsdatensatz auswendig zu lernen.

(Die genauen Zahlen ändern sich von Durchlauf zu Durchlauf, weil `make_hastie_10_2` bei jedem Aufruf einen neuen zufälligen Datensatz zieht. Worauf es ankommt, ist die Form der Kurve: ein Weak Learner, der durch Boosting zu einem Strong Learner wird.)

## Zusammenfassung

In diesem Notebook hast du:

- eine AdaBoost-Implementierung von Grund auf nachvollzogen, die auf Entscheidungsstümpfen aufbaut.
- die Aktualisierung der Gewichte verfolgt, die falsch klassifizierte Punkte neu gewichtet und jedem Weak Learner einen Score zuweist.
- den Algorithmus auf den Hastie-Benchmark angewendet und den Trainings- und Testfehler gemessen.
- visualisiert, wie die Fehlerrate von etwa 0,44 für einen einzelnen Stumpf auf rund 0,10 nach 400 Runden sinkt.

## Quellen & weiterführende Literatur

- [**Scikit-learn: AdaBoostClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html): Das fertige AdaBoost, zu dem du in der Praxis greifen würdest.
- [**Scikit-learn: Ensemble Methods**](https://scikit-learn.org/stable/modules/ensemble.html): Benutzerhandbuch, das Boosting neben Bagging, Voting und Stacking behandelt.
- [**Multi-class AdaBoosted Decision Trees**](https://scikit-learn.org/stable/auto_examples/ensemble/plot_adaboost_multiclass.html): Ein ausgearbeitetes scikit-learn-Beispiel, das AdaBoost in Aktion zeigt.
- [**AdaBoost implementation by Jaime Pastor**](https://github.com/jaimeps/adaboost-implementation): Die Originalquelle, auf der dieses Notebook aufbaut.